# Structure attention figure

Renders the four panels of the structure figure for one featured public-set complex
(`47fca019-be01-442a-878b-5e56bde1c141` — A\*03:01 / KLGGALQAK, a binder).

| Panel | Content |
|---|---|
| 1. Colored contact map | Residue contact matrix; contacts colored by interaction type. |
| 2. Raw graph | Contact-map graph; nodes colored by complex, edges by interaction. |
| 3. GCN importance | Top-25% most important nodes (node size ∝ GCN node importance). |
| 4. GAT importance | Node importance (size) with the top-5% attention edges overlaid (width ∝ attention). |

All inputs are read from `data/attention_structure/`; all shared graph/plot code lives in
`lib/structure_graph.py`. Each panel is written to `figures/structure/` as SVG + PNG.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

PUB = Path.cwd().parent            # publication/
sys.path.insert(0, str(PUB / "lib"))
DATA, FIGURES = PUB / "data", PUB / "figures"

from figure_style import apply_style, save_figure
import structure_graph as sg

apply_style()

STRUCT = DATA / "attention_structure"
IDENTIFIER = "47fca019-be01-442a-878b-5e56bde1c141"
SUBDIR = "structure"              # output goes to figures/<SUBDIR>/

## 2. Build the contact-map graph

In [ ]:
meta = pd.read_csv(STRUCT / "structure_metadata.tsv", sep="\t")
pdb_file = STRUCT / meta["pdb_file_path"].iloc[0]

# Binary C-alpha contact map -> residue graph, with one shared layout for every panel.
contact_map, residues, coords = sg.calculate_contact_map(str(pdb_file))
graph = sg.build_graph_from_contact_map(contact_map, residues, coords)
pos = sg.layout(graph)

# Assign every residue to a complex region from the chain sequences.
regions, region_coords = sg.region_assignment(meta, len(residues))

print(f"{graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
pd.Series(regions).value_counts()

## 3. Panel 1 — Colored contact map

Contact matrix as an image: each contact is colored by the interaction type of the two
residues, with labelled region separators along both axes.

In [ ]:
fig = sg.plot_colored_contact_map(contact_map, regions)
save_figure(fig, "contact_map", subdir=SUBDIR, tight=True)
plt.show()

## 4. Panel 2 — Raw graph

The contact-map graph on the shared layout: nodes colored by complex region, edges colored
by interaction type.

In [ ]:
fig = sg.plot_base_graph(graph, pos, regions)
save_figure(fig, "base_structure", subdir=SUBDIR, tight=True)
plt.show()

## 5. Panel 3 — GCN node importance

Highlight the top-25% most important nodes (size scaled by GCN node importance); edges
touching an important node keep their interaction color, the rest fade to grey.

In [ ]:
gcn = pd.read_csv(STRUCT / f"{IDENTIFIER}_attentions.tsv", sep="\t")
gcn_importance = gcn.set_index("node_index")["attention_weight"].to_dict()

fig = sg.plot_gcn_importance(graph, pos, regions, gcn_importance)
save_figure(fig, "gcn_att_structure", subdir=SUBDIR, tight=True)
plt.show()

## 6. Panel 4 — GAT node + edge importance

Node sizes scale with GAT node importance (top-25% highlighted); the top-5% attention edges
are overlaid as a directed graph whose width scales with attention, over a faint contact
backdrop.

In [ ]:
gat_nodes = pd.read_csv(STRUCT / f"{IDENTIFIER}_attention_nodes.tsv", sep="\t")
gat_edges = pd.read_csv(STRUCT / f"{IDENTIFIER}_attention_edges.tsv", sep="\t")
gat_importance = gat_nodes.set_index("node_index")["importance"].to_dict()

fig = sg.plot_gat_importance(graph, pos, regions, gat_importance, gat_edges)
save_figure(fig, "gat_att_structure", subdir=SUBDIR, tight=True)
plt.show()